# 案例 21｜订单、明细与退款三表对账：防止连接后金额翻倍

> 公开范例对照：[Looker Symmetric Aggregates](https://docs.cloud.google.com/looker/docs/best-practices/understanding-symmetric-aggregates)  
> 覆盖矩阵：多表关联膨胀 · `docs/ANALYSIS_COVERAGE.md`

## 你将学到
1. 订单↔明细 **一对多** 连接后，对订单金额 `sum` 会翻倍。
2. 正确做法：在订单粒度聚合，或对「会重复的度量」用 distinct/子查询后再汇总。
3. 加入退款表后，**销售额与退款额分别与源表对平**。
4. **wanglaoshi 缺口**：业务 D02 多表膨胀检查尚未产品化（深水区 S2）。


In [ ]:
QUICK = True
RANDOM_STATE = 42
from wanglaoshi import ready
ready(seed=RANDOM_STATE)


## 1. 合成三表：订单 / 明细 / 退款

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path
from wanglaoshi import DataAnalyzer

rng = np.random.default_rng(RANDOM_STATE)
orders = pd.DataFrame({
    'order_id': [1, 2, 3, 4],
    'order_amount': [100.0, 200.0, 150.0, 80.0],  # 订单头金额（业务源）
    'customer_id': ['c1', 'c2', 'c1', 'c3'],
})
# 一对多明细（行金额之和可与订单头一致或略有差异——课内以订单头为准对账）
items = pd.DataFrame({
    'order_id': [1, 1, 2, 2, 2, 3, 4, 4],
    'sku': ['A', 'B', 'A', 'C', 'D', 'B', 'A', 'E'],
    'line_amount': [40, 60, 50, 70, 80, 150, 30, 50],
})
refunds = pd.DataFrame({
    'refund_id': [10, 11, 12],
    'order_id': [2, 2, 4],
    'refund_amount': [20.0, 10.0, 15.0],
})
print('orders sum', orders.order_amount.sum(), 'items sum', items.line_amount.sum(), 'refunds sum', refunds.refund_amount.sum())
print(DataAnalyzer(orders).get_basic_info().get('shape'))


## 2. ❌ 错误：订单金额经明细一对多连接后求和

In [ ]:
bad = orders.merge(items, on='order_id', how='left')
print('行数膨胀', len(orders), '->', len(bad))
bad_sales = bad['order_amount'].sum()
true_sales = orders['order_amount'].sum()
print('错误汇总销售额', bad_sales, '真值', true_sales, '膨胀倍数', round(bad_sales / true_sales, 2))
assert bad_sales > true_sales, '应演示翻倍/膨胀'


## 3. ✅ 正确：先在订单粒度取不重复金额，再与退款对账

In [ ]:
# 销售额：只在订单表汇总（或 merge 后对 order_id 去重再 sum）
sales_ok = orders['order_amount'].sum()
# 若必须从宽表取：按 order_id 保留一行订单金额
dedup = bad.drop_duplicates('order_id')
sales_from_dedup = dedup['order_amount'].sum()
assert abs(sales_ok - sales_from_dedup) < 1e-9

# 退款：在退款表汇总；若 join 订单后求和退款额，同样要防一对多
ref_bad = orders.merge(refunds, on='order_id', how='left')
# 订单1、3 无退款 → NaN；订单2 两行退款
ref_wrong = ref_bad['refund_amount'].fillna(0).sum()  # 这里碰巧未膨胀订单金额，但若 sum(order_amount) 仍会错
ref_ok = refunds['refund_amount'].sum()
print('退款真值', ref_ok, '| 经订单join后对 refund 求和', ref_wrong)

# 净额（教学口径）：销售额 - 退款
net = sales_ok - ref_ok
print(dict(sales=sales_ok, refunds=ref_ok, net=net))


## 4. 对账断言：与源表勾稽

In [ ]:
out = Path('outputs/21_order_refund_reconciliation')
out.mkdir(parents=True, exist_ok=True)
recon = pd.DataFrame([{
    'metric': 'sales',
    'source': true_sales,
    'wrong_join_sum': bad_sales,
    'corrected': sales_ok,
    'balanced': abs(sales_ok - true_sales) < 1e-9,
}, {
    'metric': 'refunds',
    'source': ref_ok,
    'wrong_join_sum': ref_wrong,
    'corrected': ref_ok,
    'balanced': abs(ref_ok - refunds.refund_amount.sum()) < 1e-9,
}])
print(recon)
assert recon.loc[recon.metric == 'sales', 'balanced'].iloc[0]
assert recon.loc[recon.metric == 'refunds', 'balanced'].iloc[0]
recon.to_csv(out / 'reconciliation.csv', index=False)
print('saved', out / 'reconciliation.csv')


## 附录
- 对照 Looker：对称聚合解决的是「探索层自动防重复计数」；在 pandas ETL 里必须显式选粒度。
- 作业：把 `order_amount` 改成只存在于明细汇总，写「头表缺失时如何对账」流程。
- 包优化：`BusinessPipeline` D02 输出「膨胀倍数 + 对账差额」日志。
